In [58]:
import re

# === Analizador Léxico ===
# Definición de patrones para diferentes tipos de tokens
token_patron = {
    "KEYWORDS" : r'\b(if|else|while|return|int|float|void|then|print)\b',
    "IDENTIFIER" : r'\b[a-zA-Z][a-zA-Z0-9]*\b',
    "NUMBER" : r'\b\d+(\.\d+)?\b',
    "OPERATORS" : r'[+\-*/=<>!]',
    "DELIMITERS" : r'[(),;{}]',
    "WHITESPACE" : r'\s+'
}

def identificar_tokens(texto):
    # Unimos todos los patrones en un único patrón usando grupos nombrados
    patron_general = '|'.join(f'(?P<{token}>{patron})' for token, patron in token_patron.items())
    patron_regex = re.compile(patron_general)

    tokens_encontrados = []
    for match in patron_regex.finditer(texto):
        for token, valor in match.groupdict().items():
            if valor is not None and token != "WHITESPACE": # Ignoramos espacios en blanco
                tokens_encontrados.append((token, valor))
    return tokens_encontrados

In [59]:
class NodoAST:
    # Clase para todos los nodos del AST
    pass

    def traducirPy(self):
        # Traduccion de C++ a Python
        raise NotImplementedError("Método traducirPy() no implementado en este Nodo")
    
    def generarCodigo():
        # Traduccion de C++ a Assembler
        raise NotImplementedError("Método generarCodigo() no implementado en este Nodo")

class NodoFuncion(NodoAST):
    # Nodo que representa una función
    def __init__(self, nombre, parametros, cuerpo):
        self.nombre = nombre
        self.parametros = parametros
        self.cuerpo = cuerpo

    # Traducción a Python
    def traducirPy(self):
        params = ', '.join(p.traducirPy() for p in self.parametros)
        cuerpo = "\n    ".join(c.traducirPy() for c in self.cuerpo)
        return f"def {self.nombre[1]}({params}):\n    {cuerpo}"
    
    # Traducción a Java
    def traducirJava(self):
        params = ', '.join(p.traducirJava() for p in self.parametros)
        cuerpo = "\n    ".join(c.traducirJava() for c in self.cuerpo)
        return f"public class Programa {{\n    public static int {self.nombre[1]}({params}) {{\n        {cuerpo}\n    }}\n}}"

class NodoParametro(NodoAST):
    # Nodo que representa un parámetro de función
    def __init__(self, nombre, tipo):
        self.nombre = nombre
        self.tipo = tipo

    def traducirPy(self):
        return self.nombre[1]
    
    def traducirJava(self):
        return f"{self.tipo[1]} {self.nombre[1]}"

class NodoAsignacion(NodoAST):
    # Nodo que representa una asignación
    def __init__(self, nombre, expresion):
        self.nombre = nombre
        self.expresion = expresion

    def traducirPy(self):
        return f"{self.nombre[1]} = {self.expresion.traducirPy()}"
    
    def traducirJava(self):
        return f"{self.nombre[1]} = {self.expresion.traducirJava()};"

class NodoOperacion(NodoAST):
    # Nodo que representa una operación aritmética
    def __init__(self, izquierda, operador, derecha):
        self.izquierda = izquierda
        self.operador = operador
        self.derecha = derecha

    def traducirPy(self):
        return f"({self.izquierda.traducirPy()} {self.operador[1]} {self.derecha.traducirPy()})"
    
    def traducirJava(self):
        return f"({self.izquierda.traducirJava()} {self.operador[1]} {self.derecha.traducirJava()})"

class NodoRetorno(NodoAST):
    # Nodo que representa la sentencia de return
    def __init__(self, expresion):
        self.expresion = expresion

    def traducirPy(self):
        return f"return {self.expresion.traducirPy()}"
    
    def traducirJava(self):
        return f"return {self.expresion.traducirJava()};"

class NodoIdentificador(NodoAST):
    # Nodo que representa a un identificador
    def __init__(self, nombre):
        self.nombre = nombre

    def traducirPy(self):
        return self.nombre[1]
    
    def traducirJava(self):
        return self.nombre[1]

class NodoNumero(NodoAST):
    # Nodo que representa un número
    def __init__(self, valor):
        self.valor = valor

    def traducirPy(self):
        return str(self.valor[1])
    
    def traducirJava(self):
        return str(self.valor[1])

class NodoLlamadaFuncion(NodoAST):
    # Nodo que representa una llamada a función
    def __init__(self, nombref, argumentos):
        self.nombre_funcion = nombref
        self.argumentos = argumentos

class NodoPrint(NodoAST):
    # Nodo que representa una sentencia de impresión
    def __init__(self, expresion):
        self.expresion = expresion

    def traducirPy(self):
        return f"print({self.expresion.traducirPy()})"
    
    def traducirJava(self):
        return f"System.out.println({self.expresion.traducirJava()});"

In [60]:
# === Análisis sintáctico ===
class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def obtener_token_actual(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return None
    
    def coincidir(self, tipo_esperado):
        token_actual = self.obtener_token_actual()
        if token_actual and token_actual[0] == tipo_esperado:
            self.pos += 1
            return token_actual
        raise SyntaxError(f"Error sintáctico: se esperaba {tipo_esperado} pero se encontró: {token_actual}")

    def parse(self):
        # Punto de entrada: se espera una función
        return self.funcion()


    def funcion(self):
        self.coincidir("KEYWORDS")        # Tipo de retorno 'int'
        nombre = self.coincidir("IDENTIFIER")  # Nombre de la función
        self.coincidir("DELIMITERS")      # '('
        parametros = self.parametros()    # ← capturar la lista
        self.coincidir("DELIMITERS")      # ')'
        self.coincidir("DELIMITERS")      # '{'
        cuerpo = self.cuerpo()            # ← capturar las instrucciones
        self.coincidir("DELIMITERS")      # '}'
        return NodoFuncion(nombre, parametros, cuerpo)  # ← retornar el nodo

    def parametros(self): # Alcance privado colocar __
        # Reglas para parámetros: int IDENTIFIER (, int IDENTIFIER)*
        lista_parametros = []
        tipo = self.coincidir("KEYWORDS")  # Tipo de parámetro 'int'
        nombre = self.coincidir("IDENTIFIER")  # Nombre del parámetro
        lista_parametros.append(NodoParametro(nombre, tipo))
        while self.obtener_token_actual() and self.obtener_token_actual()[1] == ',':
            self.coincidir("DELIMITERS")  # Coma ','
            tipo = self.coincidir("KEYWORDS")  # Tipo de parámetro 'int'
            nombre = self.coincidir("IDENTIFIER")  # Nombre del parámetro
            lista_parametros.append(NodoParametro(nombre, tipo))
        return lista_parametros
    
    def cuerpo(self):
        # Gramática para el cuerpo: return IDENTIFIER OPERATOR IDENTIFIER ;
        instrucciones = []
        while self.obtener_token_actual() and self.obtener_token_actual()[1] != '}':
            if self.obtener_token_actual()[1] == 'return':
                instrucciones.append(self.retorno())
            elif self.obtener_token_actual()[1] == 'print':
                instrucciones.append(self.sentencia_print())
            else:
                instrucciones.append(self.asignacion())
        return instrucciones
    
    def sentencia_print(self):
        self.coincidir("KEYWORDS")  # 'print'
        self.coincidir("DELIMITERS") # parentesis para abrir
        expresion = self.expresion()  
        self.coincidir("DELIMITERS") # parentesis para cerrar
        self.coincidir("DELIMITERS") # punto y coma
        return NodoPrint(expresion)

    def asignacion(self):
        self.coincidir("KEYWORDS")             # Tipo 'int' o 'float'
        nombre = self.coincidir("IDENTIFIER")  # Nombre de la variable
        self.coincidir("OPERATORS")            # '='
        expresion = self.expresion()           # Lado derecho
        self.coincidir("DELIMITERS")           # ';'
        return NodoAsignacion(nombre, expresion)

    def retorno(self):
        self.coincidir("KEYWORDS")  # 'return'
        expresion = self.expresion()
        self.coincidir("DELIMITERS")  # Punto y coma ';'
        return NodoRetorno(expresion)
    
    def expresion(self):
        izquierda = self.termino()
        while self.obtener_token_actual() and self.obtener_token_actual()[0] == 'OPERATORS':
            operador = self.coincidir("OPERATORS")
            derecha = self.termino()
            izquierda = NodoOperacion(izquierda, operador, derecha)
        return izquierda
    
    def termino(self):
        token = self.obtener_token_actual()
        if token[0] == 'NUMBER':
            return NodoNumero(self.coincidir("NUMBER"))
        elif token[0] == 'IDENTIFIER':
            identificador = self.coincidir("IDENTIFIER")
            if self.obtener_token_actual() and self.obtener_token_actual()[1] == '(':
                self.coincidir("DELIMITERS")  # Paréntesis de apertura '('
                argumentos = self.llamadaFuncion()
                self.coincidir("DELIMITERS")  # Paréntesis de cierre ')'
                return NodoLlamadaFuncion(identificador[1], argumentos)
            else:
                return NodoIdentificador(identificador)
        else:
            raise SyntaxError(f"Expresión no válida: {token}")

    def llamadaFuncion(self):
        argumentos = []
        # Reglas para argumentos: IDENTIFIER | NUMBER (, IDENTIFIER | NUMBER)*
        sigue = True
        token = self.obtener_token_actual()
        while sigue:
            sigue = False
            if token[0] == 'NUMBER':
                argumento = NodoNumero(self.coincidir("NUMBER"))
            elif token[0] == 'IDENTIFIER':
                argumento = NodoIdentificador(self.coincidir("IDENTIFIER"))
            else:
                raise SyntaxError(f"Error de sintaxis, se esperaba un IDENTIFICADOR|NUMERO pero se encontró: {token}")
            argumentos.append(argumento)
            if self.obtener_token_actual() and self.obtener_token_actual()[1] == ',':
                self.coincidir("DELIMITERS")  # Coma ','
                token = self.obtener_token_actual()
                sigue = True
        return argumentos
        


In [61]:
import json

def imprimir_ast(nodo):
    if isinstance(nodo, NodoFuncion):
        return {
            'Funcion': nodo.nombre,
            'Parametros': [imprimir_ast(p) for p in nodo.parametros],
            'Cuerpo': [imprimir_ast(c) for c in nodo.cuerpo]
        }
    elif isinstance(nodo, NodoParametro):
        return {
            'Parametro': nodo.nombre,
            'Tipo': nodo.tipo
        }
    elif isinstance(nodo, NodoAsignacion):
        return {
            'Asignacion': nodo.nombre,
            'Expresion': imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoOperacion):
        return {
            'Operacion': nodo.operador,
            'Izquierda': imprimir_ast(nodo.izquierda),
            'Derecha': imprimir_ast(nodo.derecha)
        }
    elif isinstance(nodo, NodoRetorno):
        return {
            'Return': imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoIdentificador):
        return {
            'Identificador': nodo.nombre
        }
    elif isinstance(nodo, NodoNumero):
        return {
            'Numero': nodo.valor
        }
    elif isinstance(nodo, NodoPrint):
        return {
            'Print': imprimir_ast(nodo.expresion)
        }
    else: 
        return {}

In [62]:
# Ejemplo de uso
codigo_fuente = """
int suma(int a, int b) {
    int resultado = a + b;
    print(resultado);
    return resultado;
    }
"""

In [63]:
# Análisis léxico
tokens = identificar_tokens(codigo_fuente)
tokens

[('KEYWORDS', 'int'),
 ('IDENTIFIER', 'suma'),
 ('DELIMITERS', '('),
 ('KEYWORDS', 'int'),
 ('IDENTIFIER', 'a'),
 ('DELIMITERS', ','),
 ('KEYWORDS', 'int'),
 ('IDENTIFIER', 'b'),
 ('DELIMITERS', ')'),
 ('DELIMITERS', '{'),
 ('KEYWORDS', 'int'),
 ('IDENTIFIER', 'resultado'),
 ('OPERATORS', '='),
 ('IDENTIFIER', 'a'),
 ('OPERATORS', '+'),
 ('IDENTIFIER', 'b'),
 ('DELIMITERS', ';'),
 ('KEYWORDS', 'print'),
 ('DELIMITERS', '('),
 ('IDENTIFIER', 'resultado'),
 ('DELIMITERS', ')'),
 ('DELIMITERS', ';'),
 ('KEYWORDS', 'return'),
 ('IDENTIFIER', 'resultado'),
 ('DELIMITERS', ';'),
 ('DELIMITERS', '}')]

In [64]:
print("Tokens encontrados:")
for tipo, valor in tokens:
    print(f"{tipo}: {valor}")

Tokens encontrados:
KEYWORDS: int
IDENTIFIER: suma
DELIMITERS: (
KEYWORDS: int
IDENTIFIER: a
DELIMITERS: ,
KEYWORDS: int
IDENTIFIER: b
DELIMITERS: )
DELIMITERS: {
KEYWORDS: int
IDENTIFIER: resultado
OPERATORS: =
IDENTIFIER: a
OPERATORS: +
IDENTIFIER: b
DELIMITERS: ;
KEYWORDS: print
DELIMITERS: (
IDENTIFIER: resultado
DELIMITERS: )
DELIMITERS: ;
KEYWORDS: return
IDENTIFIER: resultado
DELIMITERS: ;
DELIMITERS: }


In [65]:
# Análisis sintáctico
try:
    print("\nIniciando análisis sintáctico...")
    parser = Parser(tokens)
    arbol_ast = parser.parse()
    print("Análisis sintáctico completado con éxito.")
except SyntaxError as e:
    print(e)


Iniciando análisis sintáctico...
Análisis sintáctico completado con éxito.


In [66]:
print(json.dumps(imprimir_ast(arbol_ast), indent=1))

{
 "Funcion": [
  "IDENTIFIER",
  "suma"
 ],
 "Parametros": [
  {
   "Parametro": [
    "IDENTIFIER",
    "a"
   ],
   "Tipo": [
    "KEYWORDS",
    "int"
   ]
  },
  {
   "Parametro": [
    "IDENTIFIER",
    "b"
   ],
   "Tipo": [
    "KEYWORDS",
    "int"
   ]
  }
 ],
 "Cuerpo": [
  {
   "Asignacion": [
    "IDENTIFIER",
    "resultado"
   ],
   "Expresion": {
    "Operacion": [
     "OPERATORS",
     "+"
    ],
    "Izquierda": {
     "Identificador": [
      "IDENTIFIER",
      "a"
     ]
    },
    "Derecha": {
     "Identificador": [
      "IDENTIFIER",
      "b"
     ]
    }
   }
  },
  {
   "Print": {
    "Identificador": [
     "IDENTIFIER",
     "resultado"
    ]
   }
  },
  {
   "Return": {
    "Identificador": [
     "IDENTIFIER",
     "resultado"
    ]
   }
  }
 ]
}


In [67]:
nodoExp = NodoOperacion(NodoNumero(5), '+', NodoNumero(8))
print(json.dumps(imprimir_ast(nodoExp), indent=1))

{
 "Operacion": "+",
 "Izquierda": {
  "Numero": 5
 },
 "Derecha": {
  "Numero": 8
 }
}


In [68]:
print(codigo_fuente)


int suma(int a, int b) {
    int resultado = a + b;
    print(resultado);
    return resultado;
    }



In [69]:
print(arbol_ast.traducirPy())

def suma(a, b):
    resultado = (a + b)
    print(resultado)
    return resultado


In [70]:
print(arbol_ast.traducirJava())

public class Programa {
    public static int suma(int a, int b) {
        resultado = (a + b);
    System.out.println(resultado);
    return resultado;
    }
}
